# Notebook 1: KNN Secondary Structure Prediction
**ProteinIQ Project — Concept from Lahiri et al. (2024), Ch. 22**

## Background
As described in the textbook, each residue is not classified in isolation. A **context window** of W neighboring residues is used as input to the KNN classifier (Fig 22.3). This mirrors context-based learning in humans (Fig 22.2).

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

np.random.seed(42)

## 1. Amino Acid One-Hot Encoding
Each residue is encoded as a 20-dimensional binary vector.
Within a window of W=9 residues, total features = 9 × 20 = **180**.

In [ ]:
AA_ALPHABET = 'ACDEFGHIKLMNPQRSTVWY'
AA_TO_IDX = {aa: i for i, aa in enumerate(AA_ALPHABET)}

def one_hot(aa):
    vec = np.zeros(20)
    if aa in AA_TO_IDX:
        vec[AA_TO_IDX[aa]] = 1.0
    return vec

def encode_window(sequence, i, W=9):
    half = W // 2
    features = []
    for offset in range(-half, half + 1):
        pos = i + offset
        aa = sequence[pos] if 0 <= pos < len(sequence) else '-'
        features.append(one_hot(aa))
    return np.concatenate(features)

# Visualise the window concept
seq = 'ACDEFGHIKLMNPQRSTVWY'
window_center = 5
window_vec = encode_window(seq, window_center)

fig, ax = plt.subplots(figsize=(12, 2))
ax.imshow(window_vec.reshape(9, 20), cmap='Blues', aspect='auto')
ax.set_xticks(range(20))
ax.set_xticklabels(list(AA_ALPHABET), fontsize=9)
ax.set_yticks(range(9))
ax.set_yticklabels([f'pos {i-4:+d}' for i in range(9)], fontsize=9)
ax.set_title(f'Context window (W=9) centred on residue {seq[window_center]} at position {window_center}', fontsize=11)
ax.axhline(3.5, color='red', lw=2, ls='--', label='Target residue')
ax.axhline(4.5, color='red', lw=2, ls='--')
plt.tight_layout()
plt.savefig('../data/figures/context_window.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. Chou-Fasman Propensity-Based Training Data Generation
We generate synthetic training data using Chou-Fasman propensities so the notebook runs without downloading PDB files.

In [ ]:
CF_HELIX = dict(A=1.42,L=1.21,M=1.45,F=1.13,W=1.08,K=1.16,Q=1.11,E=1.51,
                S=0.77,P=0.57,V=1.06,I=1.08,C=0.70,Y=0.69,H=1.00,D=1.01,
                N=0.67,T=0.83,G=0.57,R=0.98)
CF_STRAND = dict(A=0.83,L=1.30,M=1.05,F=1.38,W=1.37,K=0.74,Q=1.10,E=0.37,
                 S=0.75,P=0.55,V=1.70,I=1.60,C=1.19,Y=1.47,H=0.87,D=0.54,
                 N=0.89,T=1.19,G=0.75,R=0.93)

def generate_sequence(n=50):
    return ''.join(np.random.choice(list(AA_ALPHABET)) for _ in range(n))

def cf_label(seq, win=4):
    labels = []
    for i, aa in enumerate(seq):
        w = seq[max(0,i-win):i+win+1]
        ph = np.mean([CF_HELIX.get(a, 1.0) for a in w])
        pe = np.mean([CF_STRAND.get(a, 1.0) for a in w])
        if ph > 1.03 and ph > pe:  labels.append('H')
        elif pe > 1.05:             labels.append('E')
        else:                       labels.append('C')
    return ''.join(labels)

# Generate 300 synthetic training sequences
sequences = [generate_sequence(60) for _ in range(300)]
labels    = [cf_label(s) for s in sequences]

print(f'Training sequences : {len(sequences)}')
print(f'Sample sequence    : {sequences[0][:30]}...')
print(f'Sample labels      : {labels[0][:30]}...')
print(f'Label distribution : H={sum(l.count("H") for l in labels)}, '
      f'E={sum(l.count("E") for l in labels)}, '
      f'C={sum(l.count("C") for l in labels)}')

## 3. Build Feature Matrix & Train KNN

In [ ]:
X_list, y_list = [], []
for seq, lab in zip(sequences, labels):
    for i in range(len(seq)):
        X_list.append(encode_window(seq, i))
        y_list.append(lab[i])

X = np.array(X_list)   # shape: (n_residues, 180)
y = np.array(y_list)

print(f'Feature matrix shape : {X.shape}')
print(f'Unique classes       : {np.unique(y)}')

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

knn = KNeighborsClassifier(n_neighbors=5, metric='euclidean', n_jobs=-1)
knn.fit(X_train, y_train)
print(f'Training complete. n_neighbors=5, metric=euclidean')

## 4. Evaluate — Q3 Accuracy & Confusion Matrix

In [ ]:
y_pred = knn.predict(X_test)
print('Classification Report:')
print(classification_report(y_test, y_pred, target_names=['C','E','H']))

cm = confusion_matrix(y_test, y_pred, labels=['H','E','C'])
fig, ax = plt.subplots(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['H','E','C'], yticklabels=['H','E','C'])
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
ax.set_title('KNN Confusion Matrix (Q3 accuracy)')
plt.tight_layout()
plt.show()
q3 = (cm[0,0]+cm[1,1]+cm[2,2])/cm.sum()
print(f'Q3 accuracy: {q3:.3f}')

## 5. Predict a New Sequence

In [ ]:
test_seq = 'ALANMKQEHLVPGRSIWYFD'
X_new = np.array([encode_window(test_seq, i) for i in range(len(test_seq))])
pred  = knn.predict(X_new)
proba = knn.predict_proba(X_new)

print(f'Sequence  : {test_seq}')
print(f'Predicted : {" ".join(pred)}')

colors = {'H':'#D85A30','E':'#378ADD','C':'#888780'}
fig, axes = plt.subplots(2, 1, figsize=(12, 3.5))
for i, (aa, ss) in enumerate(zip(test_seq, pred)):
    axes[0].text(i, 0.5, aa, ha='center', va='center', fontsize=12, fontweight='bold')
    axes[0].add_patch(mpatches.FancyBboxPatch((i-0.4, 0.05), 0.8, 0.4,
                      boxstyle='round,pad=0.05', color=colors[ss], alpha=0.8))
    axes[0].text(i, 0.2, ss, ha='center', va='center', fontsize=9, color='white', fontweight='bold')
axes[0].set_xlim(-0.6, len(test_seq)-0.4)
axes[0].set_ylim(0,1)
axes[0].axis('off')
axes[0].set_title('Predicted secondary structure', pad=4)

classes = knn.classes_
for j, cls in enumerate(classes):
    axes[1].bar(range(len(test_seq)), proba[:,j], label=cls, alpha=0.7,
                color=colors[cls], bottom=proba[:,:j].sum(axis=1) if j>0 else 0)
axes[1].set_xlim(-0.6, len(test_seq)-0.4)
axes[1].set_xticks(range(len(test_seq)))
axes[1].set_xticklabels(list(test_seq))
axes[1].set_ylabel('Class probability')
axes[1].legend(loc='upper right', fontsize=9)
axes[1].set_title('Per-residue class probabilities')
plt.tight_layout()
plt.show()